In [1]:
import pandas as pd
import numpy as np
import os
from funciones import asignar_subnivel_por_categoria, rellenar_categorias_faltantes, consolidar_categoria, recalcular_nivel, get_level

In [2]:
product = pd.read_csv("../Datos/Originales/Datos_looks/product_2.csv", sep=";")
product_variant = pd.read_csv("../Datos/Originales/Datos_looks/product_variant.csv")
color = pd.read_csv("../Datos/Originales/Datos_looks/color.csv")
brand = pd.read_csv("../Datos/Originales/Datos_looks/brand.csv")
feature = pd.read_csv("../Datos/Originales/Datos_looks/feature.csv")
feature_value = pd.read_csv("../Datos/Originales/Datos_looks/feature_value.csv")
product_feature_value = pd.read_csv("../Datos/Originales/Datos_looks/product_feature_value.csv")

In [3]:
#Unimos: ProductFeatureValue -> FeatureValue -> Feature
pfv_fv = pd.merge(product_feature_value, feature_value, left_on="feature_value_id", right_on="id", how="inner")
full_feats = pd.merge(pfv_fv, feature, left_on="feature_id", right_on="id", how="inner", suffixes=("_val", "_feat"))

feat_df = full_feats[["product_id", "name", "value"]].copy()
feat_df.columns = ["id_producto", "nombre_feature", "valor"]

val_map = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}
mask_adv = feat_df['nombre_feature'] == 'adventurous'
feat_df.loc[mask_adv, 'valor'] = feat_df.loc[mask_adv, 'valor'].map(val_map)

feat_df.loc[feat_df['valor'].isin(['jumpsuit', 'pinafore', 'dungaree']), 'nombre_feature'] = 'jump_suit_type'
feat_df.loc[feat_df['valor'].isin(['sweatshirt']), 'nombre_feature'] = 'top_type'

features_pivoted = feat_df.pivot_table(
    index="id_producto", 
    columns="nombre_feature", 
    values="valor", 
    aggfunc="first"
).reset_index()


In [4]:
# Unimos Product + Brand
prod_brand = pd.merge(product, brand, left_on="brand_id", right_on="id", how="left", suffixes=("_prod", "_brand"))

# Unimos Merge + Variant (Usamos id_prod ya que id original cambió de nombre)
prod_var = pd.merge(prod_brand, product_variant, left_on="id_prod", right_on="product_id", how="inner")

# Unimos Merge + Color
full_df = pd.merge(prod_var, color, left_on="color_id", right_on="id", how="left", suffixes=("_brand_prod", "_color"))

rename_map = {
    "title": "nombre_producto",
    "name": "nombre_marca", 
    "name_brand": "nombre_marca", 
    "name_brand_prod": "nombre_marca",
    "name_color": "nombre_color",
    "season": "temporada"
}
full_df = full_df.rename(columns=rename_map)

if "id_producto" not in full_df.columns:
    if "id_prod" in full_df.columns:
        full_df["id_producto"] = full_df["id_prod"]
    elif "product_id" in full_df.columns:
        full_df["id_producto"] = full_df["product_id"]

cols_base = ["id_producto", "nombre_producto", "nombre_marca", "temporada", "nombre_color", "hexadecimal"]

cols_existentes = [c for c in cols_base if c in full_df.columns]
base_df = full_df[cols_existentes].drop_duplicates()
variantes_final = pd.merge(base_df, features_pivoted, on="id_producto", how="left")

In [5]:

# type_map = {
#     "top_type": "top", 
#     "jump_suit_type": "jumpsuit", 
#     "skirt_type": "skirt", 
#     "dress_type": "dress", 
#     "bag_type": "bag", 
#     "outside_type": "outerwear", 
#     "foulard_type": "scarf", 
#     "down_part_type": "bottoms" 
# }


if not variantes_final.empty:
    variantes_final[["categoria_prenda", "subcategoria_prenda"]] = variantes_final.apply(consolidar_categoria, axis=1)

    #Calcular el nivel basado en la categoría 
    variantes_final["nivel"] = variantes_final["categoria_prenda"].apply(get_level)


In [6]:
features_a_conservar = ["adventurous", "weather", "style", "application", "print", "basic"]

columnas_finales_deseadas = [
    "id_producto", "nombre_producto", "nombre_marca", "temporada", "hexadecimal",
    "categoria_prenda", "subcategoria_prenda", "nivel"
] + features_a_conservar

cols_reales = [c for c in columnas_finales_deseadas if c in variantes_final.columns]

df_resultado = variantes_final[cols_reales].copy()

if 'basic' in df_resultado.columns:
    # 1. Mapear los valores de texto a Booleanos reales
    mapa_basic = {'t': True, 'f': False, 'true': True, 'false': False}
    df_resultado['basic'] = df_resultado['basic'].map(mapa_basic)
    
    # 2. Rellenar los vacíos (NaN) con False y asegurar tipo booleano
    df_resultado['basic'] = df_resultado['basic'].fillna(False).astype(bool)
    
# Cálculo inicial del subnivel
df_resultado["subnivel"] = df_resultado.apply(asignar_subnivel_por_categoria, axis=1)

# Verificación rápida
print(df_resultado.columns)
print(df_resultado.head())
print("Total filas:", len(df_resultado))
print("NAs iniciales:\n", df_resultado.isna().sum())

C:\Users\Martin\AppData\Local\Temp\ipykernel_24864\3832898527.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_resultado['basic'] = df_resultado['basic'].fillna(False).astype(bool)


Index(['id_producto', 'nombre_producto', 'nombre_marca', 'temporada',
       'hexadecimal', 'categoria_prenda', 'subcategoria_prenda', 'nivel',
       'adventurous', 'weather', 'style', 'application', 'print', 'basic',
       'subnivel'],
      dtype='object')
                            id_producto         nombre_producto nombre_marca  \
0  5f5c4a50-557a-4d53-9e81-cc72fdd18fbf  Onljade Cardigan knit      BRAND119   
1  5f5c4a50-557a-4d53-9e81-cc72fdd18fbf  Onljade Cardigan knit      BRAND119   
2  5f5c4a50-557a-4d53-9e81-cc72fdd18fbf  Onljade Cardigan knit      BRAND119   
3  6e6b4255-006a-4d13-b312-3b1426290d8e       Poly Pant deeluxe     BRAND117   
4  6549c1b1-989e-432a-bdc9-b603c438b310     Onleden Sweater knt     BRAND119   

   temporada hexadecimal categoria_prenda subcategoria_prenda  nivel  \
0          8      000000              top            cardigan    3.0   
1          8      DFC8B2              top            cardigan    3.0   
2          8      B94600              top 

In [7]:
df_resultado.columns


Index(['id_producto', 'nombre_producto', 'nombre_marca', 'temporada',
       'hexadecimal', 'categoria_prenda', 'subcategoria_prenda', 'nivel',
       'adventurous', 'weather', 'style', 'application', 'print', 'basic',
       'subnivel'],
      dtype='object')

In [8]:
df_resultado.head()

,id_producto,nombre_producto,nombre_marca,temporada,hexadecimal,categoria_prenda,subcategoria_prenda,nivel,adventurous,weather,style,application,print,basic,subnivel
0,5f5c4a50-557a-4d53-9e81-cc72fdd18fbf,Onljade Cardigan knit,BRAND119,8,000000,top,cardigan,3.0,2,cold_season,boho,freetime,smooth,False,3.1
1,5f5c4a50-557a-4d53-9e81-cc72fdd18fbf,Onljade Cardigan knit,BRAND119,8,DFC8B2,top,cardigan,3.0,2,cold_season,boho,freetime,smooth,False,3.1
2,5f5c4a50-557a-4d53-9e81-cc72fdd18fbf,Onljade Cardigan knit,BRAND119,8,B94600,top,cardigan,3.0,2,cold_season,boho,freetime,smooth,False,3.1
3,6e6b4255-006a-4d13-b312-3b1426290d8e,Poly Pant deeluxe,BRAND117,7,B11730,bottoms,cigarette,2.0,3,warm_season,casual,freetime,smooth,False,2.2
4,6549c1b1-989e-432a-bdc9-b603c438b310,Onleden Sweater knt,BRAND119,9,FFFF00,top,sweaters,3.0,2,warm_season,classic,work,smooth,False,3.1


In [9]:
df_resultado.isna().sum()

id_producto              0
nombre_producto          0
nombre_marca             0
temporada                0
hexadecimal              0
categoria_prenda       169
subcategoria_prenda    194
nivel                  169
adventurous             80
weather                 80
style                   80
application             80
print                   93
basic                    0
subnivel               169
dtype: int64

reordenamos las columnas para un mejor manejo


In [10]:
df_resultado.isna().sum()

id_producto              0
nombre_producto          0
nombre_marca             0
temporada                0
hexadecimal              0
categoria_prenda       169
subcategoria_prenda    194
nivel                  169
adventurous             80
weather                 80
style                   80
application             80
print                   93
basic                    0
subnivel               169
dtype: int64

In [11]:
# mirar filas na
df_resultado[df_resultado.isna().any(axis=1)]

,id_producto,nombre_producto,nombre_marca,temporada,hexadecimal,categoria_prenda,subcategoria_prenda,nivel,adventurous,weather,style,application,print,basic,subnivel
259,48362d25-0918-4456-8ece-a60922e8bc6d,Claudia Scarf check,BRAND81,8,800000,NaN,NaN,NaN,2,cold,boho,work,checked,False,NaN
442,74343f9b-e16c-4572-ba97-a84ea5434166,Windsor Scarf checks,BRAND117,8,B94600,NaN,NaN,NaN,2,cold,boho,freetime,checked,False,NaN
446,d1d9ce51-920e-4a76-8442-6f51766b0225,Savannah Shirt nude,BRAND133,9,FFFFFF,top,shirt,3.0,2,warm,boho,work,NaN,False,3.1
475,4e5e9e48-2c53-499f-89d7-02c688e3a420,Filfirm Scarf wool,BRAND31,8,00008b,NaN,NaN,NaN,2,cold_season,boho,freetime,miniprint,False,NaN
476,d31746f1-e96e-4f79-b3e8-ba56e76455f6,Filoslo Scarf wool,BRAND31,8,B94600,NaN,NaN,NaN,3,cold_season,boho,freetime,horizontal_stripes,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5481,abf3fe92-f502-4091-96bd-cd5823b24a70,Objlanie Shirt denim,BRAND101,9,000081,top,shirt,3.0,2,cold_season,street,freetime,NaN,False,3.1
5541,100519f6-51be-4b37-ac9b-4ad98102a482,Nelly Cardigan bohck,BRAND97,9,164A0A,top,cardigan,3.0,2,cold_season,boho,work,NaN,False,3.1
5542,100519f6-51be-4b37-ac9b-4ad98102a482,Nelly Cardigan bohck,BRAND97,9,DFC8B2,top,cardigan,3.0,2,cold_season,boho,work,NaN,False,3.1
5635,3edbd22d-5e55-4b92-89f8-6120caa4d1d4,Pcjenna Scarf long,BRAND142,8,5F5E5E,NaN,NaN,NaN,2,cold,casual,freetime,army,False,NaN


rellenamos NAs de categoria, subcategoria en base a su nombre

In [12]:
df_resultado[['categoria_prenda', 'subcategoria_prenda']] = df_resultado.apply(rellenar_categorias_faltantes, axis=1)
df_resultado["nivel"] = df_resultado["categoria_prenda"].apply(recalcular_nivel)
df_resultado["subnivel"] = df_resultado.apply(asignar_subnivel_por_categoria, axis=1)

print("NAs restantes en categoria_prenda:", df_resultado['categoria_prenda'].isna().sum())
print("NAs restantes en subnivel:", df_resultado['subnivel'].isna().sum())

NAs restantes en categoria_prenda: 0
NAs restantes en subnivel: 0


In [13]:
df_resultado.isna().sum()

id_producto             0
nombre_producto         0
nombre_marca            0
temporada               0
hexadecimal             0
categoria_prenda        0
subcategoria_prenda    25
nivel                   0
adventurous            80
weather                80
style                  80
application            80
print                  93
basic                   0
subnivel                0
dtype: int64

In [14]:
df_resultado[df_resultado['categoria_prenda'] == 'top']['subcategoria_prenda'].unique()

array(['cardigan', 'sweaters', 'shirt', 't-shirts', 'tops', 'hoodies',
       'blouse', 'sweatshirt', nan], dtype=object)

In [15]:
#mirar nas en subcategoria
df_resultado[df_resultado['subcategoria_prenda'].isna()].head()

,id_producto,nombre_producto,nombre_marca,temporada,hexadecimal,categoria_prenda,subcategoria_prenda,nivel,adventurous,weather,style,application,print,basic,subnivel
1367,a99551fc-9d2a-4d39-a61a-1be1b3e62fdd,Akoz Top solid,BRAND73,7,00008b,top,NaN,3,NaN,NaN,NaN,NaN,NaN,False,3.1
1368,a99551fc-9d2a-4d39-a61a-1be1b3e62fdd,Akoz Top solid,BRAND73,7,D84936,top,NaN,3,NaN,NaN,NaN,NaN,NaN,False,3.1
1369,a99551fc-9d2a-4d39-a61a-1be1b3e62fdd,Akoz Top solid,BRAND73,7,000000,top,NaN,3,NaN,NaN,NaN,NaN,NaN,False,3.1
1370,a99551fc-9d2a-4d39-a61a-1be1b3e62fdd,Akoz Top solid,BRAND73,7,666633,top,NaN,3,NaN,NaN,NaN,NaN,NaN,False,3.1
1371,a99551fc-9d2a-4d39-a61a-1be1b3e62fdd,Akoz Top solid,BRAND73,7,008000,top,NaN,3,NaN,NaN,NaN,NaN,NaN,False,3.1


como la mayoria de NAS de subcategoria solo estan en Akoz Top solid	importamos 


ademas como la mayoria de nas de adventurous, weather, style, application y print estan en 3 prendas:Akoz Top solid, Kate Jacket knit y Laura Jacket paris. Entonces se les asigna un valor a cada feature y se importa

In [16]:
valores_imputados_manuales = {
    # Producto: Akoz Top solid
    'a99551fc-9d2a-4d39-a61a-1be1b3e62fdd': {
        'subcategoria_prenda': 'blouse',
        'adventurous': 3,
        'weather': 'warm_season', 
        'style': 'casual',
        'application': 'freetime',
        'print': 'smooth'
    },
    
    # Producto: Kate Jacket knit
    '1551622e-aaf6-4d6f-b80c-3048f08221b6': {
        'adventurous': 2,
        'weather': 'mid_season',
        'style': 'classic',
        'application': 'freetime',
        'print': 'smooth'
    },
    
    # Producto: Laura Jacket paris
    '3fbd87f9-e8a7-4533-bf0b-f35906cc47ee': {
        'adventurous': 4,
        'weather': 'mid_season',
        'style': 'smart',
        'application': 'work',
        'print': 'smooth'
    }
}

for id_producto, features in valores_imputados_manuales.items():
    mask = df_resultado['id_producto'] == id_producto
    
    for feature_name, feature_value in features.items():
        if feature_name in df_resultado.columns:
            if feature_name == 'adventurous':
                df_resultado.loc[mask, feature_name] = df_resultado.loc[mask, feature_name].fillna(feature_value)
            else:
                mask_na = mask & df_resultado[feature_name].isna()
                if feature_name == 'subcategoria_prenda':
                    df_resultado.loc[mask, feature_name] = feature_value
                else:
                    df_resultado.loc[mask_na, feature_name] = feature_value

C:\Users\Martin\AppData\Local\Temp\ipykernel_24864\328665833.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_resultado.loc[mask, feature_name] = df_resultado.loc[mask, feature_name].fillna(feature_value)
C:\Users\Martin\AppData\Local\Temp\ipykernel_24864\328665833.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_resultado.loc[mask, feature_name] = df_resultado.loc[mask, feature_name].fillna(feature_value)
C:\Users\Martin\AppData\Local\Temp\ipykernel_24864\328665833.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .

In [17]:

df_resultado["nivel"] = df_resultado["categoria_prenda"].apply(recalcular_nivel)
df_resultado["subnivel"] = df_resultado.apply(asignar_subnivel_por_categoria, axis=1)
if 'adventurous' in df_resultado.columns:
    df_resultado['adventurous'] = df_resultado['adventurous'].astype('Int64')

df_resultado.dropna(inplace=True)
orden_columnas = [
    'id_producto', 
    'nombre_producto', 
    'nombre_marca', 
    'temporada', 
    'hexadecimal',
    'categoria_prenda', 
    'subcategoria_prenda', 
    'adventurous', 
    'weather', 
    'style', 
    'application', 
    'print',
    'basic', 
    'nivel', 
    'subnivel' 
]

orden_final = [c for c in orden_columnas if c in df_resultado.columns]
df_resultado = df_resultado[orden_final]


In [18]:
df_resultado.isna().sum()

id_producto            0
nombre_producto        0
nombre_marca           0
temporada              0
hexadecimal            0
categoria_prenda       0
subcategoria_prenda    0
adventurous            0
weather                0
style                  0
application            0
print                  0
basic                  0
nivel                  0
subnivel               0
dtype: int64

Como quedan pocos NA, se dispone a eliminarlos

In [19]:
df_resultado.dropna(inplace=True)

In [20]:
df_resultado.isna().sum()

id_producto            0
nombre_producto        0
nombre_marca           0
temporada              0
hexadecimal            0
categoria_prenda       0
subcategoria_prenda    0
adventurous            0
weather                0
style                  0
application            0
print                  0
basic                  0
nivel                  0
subnivel               0
dtype: int64

In [21]:
os.makedirs('../Datos/Transformados', exist_ok=True)

In [22]:
#añadir un indice a df_resultado para que en el grafo cada nodo sea una variante de prenda con su color unico
df_final = df_resultado.reset_index(drop=True)
df_final['indice'] = df_final.index
columnas_ordenadas = [
    'indice', 'id_producto', 'nombre_producto', 'nombre_marca', 'temporada', 
    'hexadecimal', 'categoria_prenda', 'subcategoria_prenda', 'adventurous', 
    'weather', 'style', 'application', 'print', 'basic', 'nivel', 'subnivel'
]
df_final = df_final[columnas_ordenadas]

df_final.head()

,indice,id_producto,nombre_producto,nombre_marca,temporada,hexadecimal,categoria_prenda,subcategoria_prenda,adventurous,weather,style,application,print,basic,nivel,subnivel
0,0,5f5c4a50-557a-4d53-9e81-cc72fdd18fbf,Onljade Cardigan knit,BRAND119,8,000000,top,cardigan,2,cold_season,boho,freetime,smooth,False,3,3.1
1,1,5f5c4a50-557a-4d53-9e81-cc72fdd18fbf,Onljade Cardigan knit,BRAND119,8,DFC8B2,top,cardigan,2,cold_season,boho,freetime,smooth,False,3,3.1
2,2,5f5c4a50-557a-4d53-9e81-cc72fdd18fbf,Onljade Cardigan knit,BRAND119,8,B94600,top,cardigan,2,cold_season,boho,freetime,smooth,False,3,3.1
3,3,6e6b4255-006a-4d13-b312-3b1426290d8e,Poly Pant deeluxe,BRAND117,7,B11730,bottoms,cigarette,3,warm_season,casual,freetime,smooth,False,2,2.2
4,4,6549c1b1-989e-432a-bdc9-b603c438b310,Onleden Sweater knt,BRAND119,9,FFFF00,top,sweaters,2,warm_season,classic,work,smooth,False,3,3.1


In [23]:
df_final["indice"].unique()

array([   0,    1,    2, ..., 5672, 5673, 5674], shape=(5675,))

In [24]:
len(df_final)

5675

In [25]:
df_final.to_csv("../Datos/Transformados/df_resultado.csv", sep=';',index=False)